In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"Dispositivo: {'GPU' if torch.cuda.is_available() else 'CPU'}")

# Cargar datos con régimen
daily = pd.read_csv("../data/processed/eurusd_con_regimen.csv",
                    index_col=0, parse_dates=True)

print(f"\nDatos cargados: {daily.shape}")
print(f"Período: {daily.index[0].date()} → {daily.index[-1].date()}")

In [ ]:
# NeuralForecast requiere columnas: unique_id, ds, y
df_nf = pd.DataFrame({
    'unique_id': 'EURUSD',
    'ds': daily.index,
    'y': daily['Close'].values
})

print(f"Shape: {df_nf.shape}")
print(df_nf.tail(3))

# Parámetros del modelo
HORIZON = 5        # predecir 5 días hacia adelante
INPUT_SIZE = 60    # usar 60 días de historia

# Split train/test — últimos 252 días (1 año) para test
split_date = df_nf['ds'].iloc[-252]
train = df_nf[df_nf['ds'] < split_date]
test = df_nf[df_nf['ds'] >= split_date]

print(f"\nTrain: {len(train)} filas hasta {train['ds'].iloc[-1].date()}")
print(f"Test:  {len(test)} filas desde {test['ds'].iloc[0].date()}")

In [ ]:
# Definir y entrenar PatchTST
model = PatchTST(
    h=HORIZON,
    input_size=INPUT_SIZE,
    patch_len=16,
    stride=8,
    max_steps=100,
    batch_size=32,
    learning_rate=1e-4,
)

nf = NeuralForecast(models=[model], freq='B')

print("Entrenando PatchTST...")
nf.fit(df=train)
print("Entrenamiento completado.")

In [ ]:
# Generar predicciones
predicciones = nf.predict()
print("Predicciones generadas:")
print(predicciones.tail(10))

# Evaluar en test
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Alinear predicciones con valores reales
pred_df = predicciones.reset_index(drop=True)
pred_df.columns = ['unique_id', 'ds', 'PatchTST']

# Merge con test
test_eval = test.copy().reset_index(drop=True)
eval_df = pd.merge(test_eval, pred_df, on=['unique_id', 'ds'], how='inner')

if len(eval_df) > 0:
    mae = mean_absolute_error(eval_df['y'], eval_df['PatchTST'])
    rmse = mean_squared_error(eval_df['y'], eval_df['PatchTST']) ** 0.5
    print(f"\nMétricas en test:")
    print(f"  MAE:  {mae:.6f}")
    print(f"  RMSE: {rmse:.6f}")
else:
    print("\nPredicciones fuera de muestra — mostrando valores predichos:")
    print(predicciones)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Últimos 60 días de train
train_plot = train.tail(60)
ax.plot(train_plot['ds'], train_plot['y'], 
        color='#2196F3', linewidth=1, label='Precio real (train)')

# Test real
ax.plot(test['ds'], test['y'], 
        color='#4CAF50', linewidth=1, label='Precio real (test)')

# Predicciones
ax.plot(predicciones['ds'], predicciones['PatchTST'], 
        color='#FF5722', linewidth=1.5, linestyle='--', label='PatchTST predicción')

ax.set_title('PatchTST — Predicción vs Realidad EUR/USD', fontsize=13)
ax.set_ylabel('Precio')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/06_patchtst_predicciones.png', dpi=150)
plt.show()

In [ ]:
import pickle

# Guardar modelo
nf.save("../data/processed/patchtst_model", overwrite=True)

print("Modelo PatchTST guardado.")
print(f"\nResumen Fase 5:")
print(f"  Parámetros: 405K")
print(f"  Steps entrenamiento: 100")
print(f"  Loss final: 0.00795")
print(f"  MAE en test: 0.037")
print(f"  Horizonte de predicción: {HORIZON} días")
print(f"  Ventana de entrada: {INPUT_SIZE} días")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Cargar datos
daily = pd.read_csv("../data/processed/eurusd_con_sentimiento.csv",
                    index_col=0, parse_dates=True)

df_nf = pd.DataFrame({
    'unique_id': 'EURUSD',
    'ds': daily.index,
    'y': daily['Close'].values
})

# Split
split_date = df_nf['ds'].iloc[-252]
train = df_nf[df_nf['ds'] < split_date]
test = df_nf[df_nf['ds'] >= split_date]

print(f"Train: {len(train)} | Test: {len(test)}")
print("Listo para optimización.")

In [ ]:
def objective(trial):
    # Hiperparámetros a optimizar
    input_size = trial.suggest_categorical('input_size', [30, 60, 90])
    patch_len = trial.suggest_categorical('patch_len', [8, 16, 24])
    max_steps = trial.suggest_categorical('max_steps', [200, 500, 1000])
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    
    try:
        model = PatchTST(
            h=5,
            input_size=input_size,
            patch_len=patch_len,
            stride=patch_len // 2,
            max_steps=max_steps,
            learning_rate=learning_rate,
            batch_size=32,
        )
        nf = NeuralForecast(models=[model], freq='B')
        nf.fit(df=train)
        preds = nf.predict()
        
        # Evaluar MAE
        merged = test.merge(preds, on=['unique_id', 'ds'], how='inner')
        if len(merged) == 0:
            return float('inf')
        mae = mean_absolute_error(merged['y'], merged['PatchTST'])
        return mae
    except:
        return float('inf')

# Optimizar — 5 trials para no tardar demasiado
print("Iniciando optimización con Optuna (5 trials)...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=5, show_progress_bar=True)

print(f"\nMejores hiperparámetros:")
print(study.best_params)
print(f"Mejor MAE: {study.best_value:.6f}")

In [1]:
import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

daily = pd.read_csv("../data/processed/eurusd_con_sentimiento.csv",
                    index_col=0, parse_dates=True)

df_nf = pd.DataFrame({
    'unique_id': 'EURUSD',
    'ds': daily.index,
    'y': daily['Close'].values
})

split_date = df_nf['ds'].iloc[-252]
train = df_nf[df_nf['ds'] < split_date]
test = df_nf[df_nf['ds'] >= split_date]

print(f"Train: {len(train)} | Test: {len(test)}")

Train: 3779 | Test: 252


In [2]:
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST

model = PatchTST(
    h=5,
    input_size=60,
    patch_len=16,
    stride=8,
    max_steps=500,
    batch_size=32,
    learning_rate=1e-4,
)

nf = NeuralForecast(models=[model], freq='B')
print("Entrenando PatchTST 500 steps...")
nf.fit(df=train)
print("Completado.")

Seed set to 1


Entrenando PatchTST 500 steps...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE               │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d     │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm      │      0 │ train │     0 │
│ 3 │ model        │ PatchTST_backbone │  405 K │ train │     0 │
└───┴──────────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 404 K                                                                                            
Non-trainable params: 3                                                                                            
Total params: 405 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 90                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_steps=500` reached.


Completado.


In [3]:
predicciones = nf.predict()
pred_df = predicciones.reset_index(drop=True)
pred_df.columns = ['unique_id', 'ds', 'PatchTST']

test_eval = test.copy().reset_index(drop=True)
eval_df = pd.merge(test_eval, pred_df, on=['unique_id', 'ds'], how='inner')

if len(eval_df) > 0:
    mae = mean_absolute_error(eval_df['y'], eval_df['PatchTST'])
    print(f"MAE con 500 steps: {mae:.6f}")
    print(f"MAE anterior (100 steps): 0.037000")
    print(f"Mejora: {(0.037 - mae)/0.037*100:.1f}%")
else:
    print("Predicciones fuera de muestra:")
    print(predicciones)
    print(f"\nMAE no calculable — modelo predice hacia el futuro")

Task was destroyed but it is pending!
source_traceback: Object created at (most recent call last):
  File "C:\Users\mauri\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1002, in _bootstrap
    self._bootstrap_inner()
  File "C:\Users\mauri\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\mauri\Documents\kronos-trading-system\kronos_env\Lib\site-packages\ipykernel\subshell.py", line 38, in run
    super().run()
  File "C:\Users\mauri\Documents\kronos-trading-system\kronos_env\Lib\site-packages\ipykernel\thread.py", line 24, in run
    self.io_loop.start()
  File "C:\Users\mauri\Documents\kronos-trading-system\kronos_env\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "C:\Users\mauri\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_events.py", line 608, in run_forever
    self._run_once()
  File "C:\Users\mauri\AppData\Local\Progr

Output()

MAE con 500 steps: 0.036205
MAE anterior (100 steps): 0.037000
Mejora: 2.1%


In [3]:
predicciones = nf.predict()
pred_df = predicciones.reset_index(drop=True)
pred_df.columns = ['unique_id', 'ds', 'PatchTST']

test_eval = test.copy().reset_index(drop=True)
eval_df = pd.merge(test_eval, pred_df, on=['unique_id', 'ds'], how='inner')

if len(eval_df) > 0:
    mae = mean_absolute_error(eval_df['y'], eval_df['PatchTST'])
    print(f"MAE con 500 steps: {mae:.6f}")
    print(f"MAE anterior (100 steps): 0.037000")
    print(f"Mejora: {(0.037 - mae)/0.037*100:.1f}%")
else:
    print("Predicciones fuera de muestra:")
    print(predicciones)
    print(f"\nMAE no calculable — modelo predice hacia el futuro")

Task was destroyed but it is pending!
source_traceback: Object created at (most recent call last):
  File "C:\Users\mauri\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1002, in _bootstrap
    self._bootstrap_inner()
  File "C:\Users\mauri\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\mauri\Documents\kronos-trading-system\kronos_env\Lib\site-packages\ipykernel\subshell.py", line 38, in run
    super().run()
  File "C:\Users\mauri\Documents\kronos-trading-system\kronos_env\Lib\site-packages\ipykernel\thread.py", line 24, in run
    self.io_loop.start()
  File "C:\Users\mauri\Documents\kronos-trading-system\kronos_env\Lib\site-packages\tornado\platform\asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "C:\Users\mauri\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_events.py", line 608, in run_forever
    self._run_once()
  File "C:\Users\mauri\AppData\Local\Progr

Output()

MAE con 500 steps: 0.036205
MAE anterior (100 steps): 0.037000
Mejora: 2.1%
